In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("kyc customer").getOrCreate()

storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(
    scope="aml-scope",
    key="storage-access-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    storage_key
)


df_kyc_raw = spark.read.option("header","true").option("inferSchema","true").format("csv").load("wasbs://raw@stbankamldev.blob.core.windows.net/dirty_aml_customers.csv")


df_risk_lookup = spark.read.option("header","true").option("inferSchema","true").csv("wasbs://raw@stbankamldev.blob.core.windows.net/country_risk_lookup.csv")

df_kyc_raw.show(5)
df_kyc_raw.printSchema()

df_risk_lookup.show(5)

df_risk_lookup.printSchema()

In [0]:
from pyspark.sql.functions import *

df_kyc_join = df_kyc_raw.join(df_risk_lookup, df_kyc_raw.country_code == df_risk_lookup.country_code, "left") \
    .select("customer_id","customer_name","email",df_kyc_raw["country_code"]
            ,"kyc_status","transaction_amount","transaction_type",
            df_risk_lookup["country_risk_level"])
    

df_kyc_clean = df_kyc_join \
    .withColumn("clean_email", trim(lower(col("email")))) \
    .withColumn("clean_kyc_status", trim(lower(col("kyc_status")))) \
    .withColumn("clean_country_code", trim(upper(col("country_code")))) \
    .select("customer_id","customer_name","clean_email","clean_country_code","clean_kyc_status","transaction_amount","transaction_type","country_risk_level")

display(df_kyc_clean)


In [0]:
email_pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

df_kyc_transformed = df_kyc_clean \
    .withColumn("email_valid_flag" , when(col("clean_email").rlike(email_pattern),1).otherwise(0)) \
    .withColumn("domain", when(col("clean_email").contains("@"),split(col("clean_email"),"@")[1]).otherwise("None")) \
    .withColumn("unusual_domain_flag", 
    when((col("domain").isNull()) |
        (col("domain") == "") |
        (col("domain") == "None") |
        (col("domain").isin("mail.ru","tempmail.com","unknown.xyz","secure-mail.net","protonmail.com","company.ir")),1).otherwise(0)) \
    .withColumn("kyc_incomplete_flag", when(col("clean_kyc_status") != "complete",1).otherwise(0)) \
    .withColumn("high_risk_country_flag" , when(col("country_risk_level") == ("HIGH"), 1).otherwise(0)) \
   .withColumn("risk_score" , (1 - col("email_valid_flag"))+col("unusual_domain_flag")+col("kyc_incomplete_flag")+col("high_risk_country_flag")) \
    .withColumn("risk_level", when(col("risk_score") >= 3,"HIGH")
                            .when (col("risk_score") >=2 , "MEDIUM")
                            .otherwise("LOW")) \
    .withColumn("risk_reason" , concat_ws("|",when(col("email_valid_flag") ==0 ,"EMAIL_INVALID"),
                                              when(col("unusual_domain_flag") == 1 , "UNUSUAL_DOMAIN"),
                                              when(col("kyc_incomplete_flag") == 1 , "KYC_INCOMPLETE"),
                                              when(col("high_risk_country_flag") == 1, "HIGH_RISK_COUNTRY")
                                          ))
                

df_kyc_transformed.show(20)
silver_path = "wasbs://silver@stbankamldev.blob.core.windows.net/customer_risk_assessment"

df_kyc_transformed.write.mode("overwrite").parquet('wasbs://silver@stbankamldev.blob.core.windows.net/customer_risk_assessment')


In [0]:
df_silver_kyc = spark.read.parquet(silver_path)
display(df_silver_kyc)

In [0]:
df_gold_customer_risk = df_silver_kyc \
    .select("customer_id","customer_name","clean_country_code","country_risk_level","risk_score","risk_level","risk_reason")

df_gold_customer_risk.write.mode("overwrite").parquet("wasbs://gold@stbankamldev.blob.core.windows.net/customer_risk_score")    

df_gold_customer_risk.show(20)

In [0]:
df_gold_customer_risk.groupBy("risk_level").count().show()